# Review title + final full-text criteria (split questions)

**One factor changed relative to [08a_split_questions.ipynb](08a_split_questions.ipynb)** (entry 8): `state.review.title` holds the review title. Everything else is identical: four `noul` questions in one call, one per section of the final review's *Criteria for considering studies*, same question sentence, record title + abstract in `state.record`, same combinations (product of all four; product without outcomes).

Tests how much the topic given by the title helps the per-section questions (the full-protocol run suggested the sections lack the review's topic).

Labels: CLEF TAR abstract-level (`label_abs`) and content-level (`label_content`).

Nothing paid runs unless `RUN_SMOKE`, `RUN_SUBSET` or `RUN_PAID` is set to `True` below.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
import screening as S

RUN_SMOKE = False    # one paid call
RUN_SUBSET = True    # the 10 reviews with < 1000 records (3,637 records); see the cost estimate
RUN_PAID = False     # all 20 reviews; read the cost estimate first
OUT = S.out_dir("title_fulltext_criteria_split_questions")
FINAL = S.out_dir("fulltext_criteria_split_questions")      # same questions, no title (entry 8, all 20 reviews)
REFERENCE = {"baseline": S.out_dir("baseline"), "title_abstract_no_results": S.out_dir("review_title_abstract_no_results")}

/Users/joshua/Desktop/Repositories/Research/decision-model-screener/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data and split

In [2]:
records, CRITERIA = S.load_records()
records = S.add_bm25(records, CRITERIA)
PROTO = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_protocol_criteria.json").read_text())
FT = json.loads((ROOT / "data" / "processed" / "tar2019_intervention_fulltext_criteria.json").read_text())
assert set(PROTO) == set(CRITERIA)

SECTIONS = {"studies": "Types of studies", "participants": "Types of participants",
            "interventions": "Types of interventions", "outcomes": "Types of outcome measures"}

def split(criteria):
    """{question key: section text without its heading}, split at the '#### Types of ...' headings."""
    parts, key = {}, None
    for line in criteria.split("\n")[1:]:
        head = line.lstrip("#").strip()
        if line.startswith("#### ") and head in SECTIONS.values():
            key = next(k for k, v in SECTIONS.items() if v == head)
            parts[key] = []
        else:
            parts[key].append(line)
    return {k: "\n".join(v).strip() for k, v in parts.items()}

PARTS = {r: split(v["criteria"]) for r, v in FT.items()}
FINAL_PARTS = {r: split(v["criteria"]) for r, v in FT.items()}
assert all(set(p) == set(SECTIONS) and all(p.values()) for p in PARTS.values()), "a review lacks a section"

n = records.groupby("review").size()
SUBSET = sorted(n[n < 1000].index)
print("subset:", len(SUBSET), "reviews,", int(n[SUBSET].sum()), "records")
pd.DataFrame({"title": {r: c["title"] for r, c in CRITERIA.items()}, "in_subset": {r: r in SUBSET for r in CRITERIA}})

20 reviews, 41996 records, 41183 unique PMIDs
not returned by PubMed: 0 | no abstract: 2183 of 41996


subset: 10 reviews, 3637 records


,title,in_subset
CD000996,Inhaled corticosteroids for bronchiectasis,True
CD001261,Vaccines for preventing typhoid fever,True
CD004414,Interventions for preventing occupational irri...,True
CD006468,Anticoagulation for people with cancer and cen...,False
CD007867,Prescribed hypocaloric nutrition support for c...,True
CD009069,Prophylactic vaccination against human papillo...,False
CD009642,Continuous intravenous perioperative lidocaine...,False
CD010038,Face-to-face interventions for informing or ed...,False
CD010239,Lower versus higher oxygen concentrations titr...,True
CD010558,Psychological therapies for treatment-resistan...,False


## Decision model input

Same questions as entry 8; the state adds the review title.

In [3]:
def build_body(review, title, abstract):
    return {
        "model": S.MODEL,
        "state": {"review": {"title": CRITERIA[review]["title"]},
                  "record": {"title": title, "abstract": abstract}},
        "questions": {
            k: {"type": "noul",
                "instructions": f"The record meets this eligibility criterion of the systematic review ({SECTIONS[k].lower()}): "
                                f"it meets the stated requirements and none of the stated exclusions.\n\n{PARTS[review][k]}"}
            for k in SECTIONS
        },
    }

print(json.dumps(build_body(*records.loc[0, ["review", "title", "abstract"]]), indent=1)[:4000])

{
 "model": "typesafe/jev-1.14",
 "state": {
  "review": {
   "title": "Inhaled corticosteroids for bronchiectasis"
  },
  "record": {
   "title": "Bronchiectasis Severity Is an Independent Risk Factor for Vascular Disease in a Bronchiectasis Cohort.",
   "abstract": "BACKGROUND: Interest in the association of vascular disease with COPD and pneumonia has increased, but there is a lack of research in this area with patients with bronchiectasis.\nMETHODS: A retrospective study of 400 patients attending a specialist bronchiectasis clinic in NHS Lothian (Edinburgh, UK) between May 2013 and September 2014 was conducted. The study assessed the prevalence of vascular disease (ischemic heart disease, cerebrovascular disease, peripheral vascular disease, and atrial fibrillation). Using multivariable models, independent risk factors were identified for vascular disease that developed following the diagnosis of bronchiectasis.\nRESULTS: The study included 400 patients. There was preexisting vascu

## Cost estimate (no API call)

In [4]:
est = S.estimate_cost(records, build_body)
print(f"subset ({len(SUBSET)} reviews): ${est.loc[SUBSET, 'usd'].sum():.2f}")

TypeSafe | typesafe/jev-1.14-20261014 | USD per 1M input tokens: 0.041999999999999996 | output: 0


requests: 41996 | mean chars per request: 8491 | max: 23509
estimated input tokens: 116.5M | estimated cost: $4.89
subset (10 reviews): $0.30


## Smoke test (one paid call)

In [5]:
if RUN_SMOKE:
    print(json.dumps(S.call(build_body(*records.loc[0, ["review", "title", "abstract"]])), indent=1))

## Subset (paid)

In [6]:
if RUN_SUBSET:
    S.run_reviews(records, SUBSET, build_body, OUT)

reviews:   0%|          | 0/10 [00:00<?, ?it/s]

CD000996:   0%|          | 0/281 [00:00<?, ?it/s]

CD000996:   0%|          | 1/281 [00:00<01:40,  2.79it/s]

CD000996:   2%|▏         | 7/281 [00:00<00:19, 14.17it/s]

CD000996:   4%|▍         | 12/281 [00:00<00:13, 20.39it/s]

CD000996:   6%|▌         | 16/281 [00:00<00:10, 24.13it/s]

... (976 outputs left out)

CD012551:  99%|█████████▉| 587/591 [00:23<00:00, 23.90it/s]

CD012551: 100%|█████████▉| 590/591 [00:23<00:00, 21.62it/s]

reviews: 100%|██████████| 10/10 [02:29<00:00, 13.70s/it]

reviews: 100%|██████████| 10/10 [02:29<00:00, 14.94s/it]

## Full run (paid)

In [7]:
if RUN_PAID:
    S.run_reviews(records, CRITERIA, build_body, OUT)

## Results

Protocol vs final criteria on the same records: the reviews with a cache file for this run (the final-criteria split run covers all 20). LGAR baseline and title + abstract without results for reference. A failed call or a missing answer gives the lowest score (counted below).

In [8]:
done = sorted(p.stem for p in OUT.glob("*.jsonl"))
print(len(done), "of", len(CRITERIA), "reviews run")
sub = records[records.review.isin(done)].copy()
for col, path in REFERENCE.items():
    sub = S.add_scores(sub, path, col)

def load_split(path, prefix):
    rows = []
    for r in done:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            ans = row["response"].get("answers", {})
            rows.append({"review": r, "pmid": row["pmid"], **{f"{prefix}{k}": ans.get(k, {}).get("noul") for k in SECTIONS}})
    return pd.DataFrame(rows, columns=["review", "pmid", *[prefix + k for k in SECTIONS]]).drop_duplicates(["review", "pmid"], keep="last")

for prefix, path in (("p_", OUT), ("f_", FINAL)):
    sub = sub.merge(load_split(path, prefix), on=["review", "pmid"], how="left")
    cols = [prefix + k for k in SECTIONS]
    bad = sub[cols].isna().any(axis=1)
    print(prefix, "records with a failed call or a missing answer (ranked last):", int(bad.sum()))
    sub[f"{prefix}all4"] = sub[cols].prod(axis=1).where(~bad, -1.0)
    sub[f"{prefix}no_outcomes"] = sub[cols[:3]].prod(axis=1).where(~bad, -1.0)

NAMES = {"baseline": "LGAR baseline (reference)", "title_abstract_no_results": "title + abstract w/o results (reference)",
         "f_all4": "no title, product all 4", "f_no_outcomes": "no title, without outcomes",
         "p_all4": "+ review title, product all 4", "p_no_outcomes": "+ review title, without outcomes"}
results = {NAMES[c]: S.evaluate(sub, c) for c in NAMES}
if done:
    for a in ("all4", "no_outcomes"):
        d = results[NAMES[f"p_{a}"]] - results[NAMES[f"f_{a}"]]
        d["reviews"] = results[NAMES[f"f_{a}"]]["reviews"]
        results[f"title - no title ({a})"] = d
pd.concat(results)

10 of 20 reviews run
baseline: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $1.80 | without a score (ranked last): 0


title_abstract_no_results: scored 3637 of 3637 records | model: {'typesafe/jev-1.14-20261014'} | cost: $2.06 | without a score (ranked last): 0
p_ records with a failed call or a missing answer (ranked last): 0
f_ records with a failed call or a missing answer (ranked last): 0


MAP  R@1%  R@5%  \
LGAR baseline (reference)                label_abs      70.7  21.0  60.1   
                                         label_content  77.2  31.4  79.8   
title + abstract w/o results (reference) label_abs      74.4  22.3  60.4   
                                         label_content  85.4  36.0  84.7   
no title, product all 4                  label_abs      68.5  22.8  55.2   
                                         label_content  71.4  28.4  71.9   
no title, without outcomes               label_abs      69.9  23.4  56.8   
                                         label_content  77.1  30.2  80.0   
+ review title, product all 4            label_abs      70.1  23.5  55.4   
                                         label_content  73.0  28.2  74.6   
+ review title, without outcomes         label_abs      70.4  22.1  56.8   
                                         label_content  76.8  28.4  81.1   
title - no title (all4)                  label_abs       1.6   0.7   0.2   
                                         label_content   1.6  -0.2   2.7   
title - no title (no_outcomes)           label_abs       0.5  -1.3   0.0   
                                         label_content  -0.3  -1.8   1.1   

                                                        R@10%  R@20%  R@50%  \
LGAR baseline (reference)                label_abs       73.7   84.8   96.8   
                                         label_content   88.6   94.3   99.2   
title + abstract w/o results (reference) label_abs       75.8   86.8   99.0   
                                         label_content   91.8   95.9   99.1   
no title, product all 4                  label_abs       70.0   80.0   97.5   
                                         label_content   88.3   97.0   99.7   
no title, without outcomes               label_abs       70.7   82.3   99.2   
                                         label_content   88.7   96.4   99.4   
+ review title, product all 4            label_abs       69.9   83.6   98.9   
                                         label_content   88.9   96.7   99.7   
+ review title, without outcomes         label_abs       69.4   83.3   99.2   
                                         label_content   88.3   96.4   99.7   
title - no title (all4)                  label_abs       -0.1    3.6    1.4   
                                         label_content    0.6   -0.3    0.0   
title - no title (no_outcomes)           label_abs       -1.3    1.0    0.0   
                                         label_content   -0.4    0.0    0.3   

                                                        WSS@95%  TNR@95%  \
LGAR baseline (reference)                label_abs         60.5     71.1   
                                         label_content     79.1     88.0   
title + abstract w/o results (reference) label_abs         63.8     74.8   
                                         label_content     83.0     92.0   
no title, product all 4                  label_abs         61.0     71.9   
                                         label_content     82.5     91.8   
no title, without outcomes               label_abs         61.9     72.5   
                                         label_content     82.5     91.6   
+ review title, product all 4            label_abs         63.9     75.1   
                                         label_content     83.0     92.3   
+ review title, without outcomes         label_abs         64.6     75.6   
                                         label_content     82.0     91.1   
title - no title (all4)                  label_abs          2.9      3.2   
                                         label_content      0.5      0.5   
title - no title (no_outcomes)           label_abs          2.7      3.1   
                                         label_content     -0.5     -0.5   

                                                        WSS@100%  reviews  
LGAR baseline (reference)                label_abs 

In [9]:
# per review: protocol vs final, and per-section ablations
per = {}
for r, df in sub.groupby("review"):  # empty until a run has finished
    for label in ("label_abs", "label_content"):
        for c in ("f_no_outcomes", "p_no_outcomes"):
            m = S.metrics(df.sort_values([c, "bm25", "pmid"], ascending=[False, False, True])[label])
            per.setdefault(r, {})[(label, c)] = round(m["MAP"] * 100, 1) if m else None
per = pd.DataFrame(per).T
for label in ("label_abs", "label_content") if len(per) else ():
    d = per[(label, "p_no_outcomes")] - per[(label, "f_no_outcomes")]
    print(f"{label}: with vs without title (without outcomes): better {(d > 0).sum()} / worse {(d < 0).sum()} of {d.notna().sum()} reviews")
if done:
    sec = {}
    for k in SECTIONS:
        for prefix, name in (("f_", "no title"), ("p_", "with title")):
            ev = S.evaluate(sub, prefix + k)
            sec[(k, name)] = {lab: ev.loc[lab, "MAP"] for lab in ("label_abs", "label_content")}
    display(pd.DataFrame(sec).T.rename(columns=lambda c: f"MAP {c} (section alone)"))
per

label_abs: with vs without title (without outcomes): better 6 / worse 3 of 10 reviews
label_content: with vs without title (without outcomes): better 2 / worse 4 of 9 reviews


MAP label_abs (section alone)  \
studies       no title                             37.2   
              with title                           62.2   
participants  no title                             40.5   
              with title                           46.1   
interventions no title                             60.7   
              with title                           60.9   
outcomes      no title                             29.6   
              with title                           38.7   

                          MAP label_content (section alone)  
studies       no title                                 45.0  
              with title                               78.1  
participants  no title                                 35.9  
              with title                               39.9  
interventions no title                                 63.8  
              with title                               64.3  
outcomes      no title                                 19.3  
              with title                               28.9

label_abs               label_content              
         f_no_outcomes p_no_outcomes f_no_outcomes p_no_outcomes
CD000996          79.6          79.7         100.0         100.0
CD001261          56.0          58.0          61.5          68.8
CD004414          57.3          64.0          38.2          53.2
CD007867          56.7          53.5          66.7          64.0
CD010239          92.3          92.7         100.0         100.0
CD011140          89.3          89.3           NaN           NaN
CD011571          75.0          83.4          97.6          97.6
CD011977          80.3          81.2          83.0          82.5
CD012164          47.6          38.4          68.1          47.2
CD012551          64.5          63.9          79.1          78.2

In [10]:
# usage on exactly these records
def usage(path, reviews):
    tok = cost = n = 0
    for r in reviews:
        for row in map(json.loads, (path / f"{r}.jsonl").open()):
            u = row["response"].get("usage", {})
            tok += u.get("input_tokens", 0); cost += u.get("cost") or 0; n += 1
    return {"calls": n, "input_tokens": tok, "usd": round(cost, 3), "tokens_per_call": round(tok / max(n, 1))}

pd.DataFrame({"no title": usage(FINAL, done), "+ review title": usage(OUT, done)}).T if done else None

,calls,input_tokens,usd,tokens_per_call
no title,3637.0,5460474.0,0.229,1501.0
+ review title,3637.0,5563379.0,0.234,1530.0
